这是**TOPSIS模型 (Technique for Order Preference by Similarity to an Ideal Solution)** 的详细解析。

它是数学建模中最**“耐用”**、**“好用”**、**“不挑食”**的综合评价方法。俗称“优劣解距离法”。

---

### 一、 算法原理
**核心思想**：**“近朱者赤，近墨者黑”。**

想象在一个多维空间里（比如二维平面）：
1.  我们找出一个**“最完美的点”**（正理想解 $Z^+$）：它所有指标都是最好的（比如：价格最低、性能最高、颜值最高）。
2.  我们找出一个**“最垃圾的点”**（负理想解 $Z^-$）：它所有指标都是最差的。
3.  **评分逻辑**：看你的评价对象，离“完美点”有多近，同时离“垃圾点”有多远。离完美越近、离垃圾越远，得分越高。

---

### 二、 推导与步骤

假设有 $n$ 个评价对象， $m$ 个指标。

#### 1. 指标正向化 (非常重要)
TOPSIS要求所有指标方向一致（通常统一为**越大越好**）。
*   **极小型（越小越好）**：转化为极大型。公式：$x' = \max(x) - x$。
*   **中间型（越接近某值越好）**：比如PH值=7最好。需转化为极大型（公式较复杂，代码中通常简化处理或手动预处理）。
*   **区间型（在某区间内最好）**：比如体温36-37度。

#### 2. 数据标准化 (向量归一化)
注意：TOPSIS通常使用**向量归一化**，而不是Min-Max归一化。
$$z_{ij} = \frac{x_{ij}}{\sqrt{\sum_{i=1}^{n} x_{ij}^2}}$$
这样做的好处是利用了几何距离的特性。

#### 3. 确定权重 (可选)
TOPSIS本身不产出权重。你需要给它权重向量 $w$。
*   如果题目没给：通常搭配**熵权法**或**AHP**计算权重。
*   如果不想算：默认为等权重 ($1/m$)。

#### 4. 确定正负理想解
*   **正理想解** $Z^+$：每一列的最大值向量（因为第一步已经把所有指标转成正向了）。
    $Z^+ = [Z^+_1, Z^+_2, ..., Z^+_m]$
*   **负理想解** $Z^-$：每一列的最小值向量。
    $Z^- = [Z^-_1, Z^-_2, ..., Z^-_m]$

#### 5. 计算距离 (欧氏距离)
第 $i$ 个对象与最优解的距离 $D^+_i$：
$$D^+_i = \sqrt{\sum_{j=1}^{m} w_j (Z^+_j - z_{ij})^2}$$
第 $i$ 个对象与最劣解的距离 $D^-_i$：
$$D^-_i = \sqrt{\sum_{j=1}^{m} w_j (Z^-_j - z_{ij})^2}$$

#### 6. 计算得分
$$S_i = \frac{D^-_i}{D^+_i + D^-_i}$$
*   $0 \le S_i \le 1$。
*   如果 $S_i$ 接近 1，说明 $D^-$ 很大（离垃圾很远），$D^+$ 很小（离完美很近）。

---

### 三、 适用场景
1.  **综合评价**：医院医疗质量评价、水质评价、企业财务状况评价。
2.  **数据量灵活**：对样本数量没有严格限制（不像回归分析需要大量数据）。
3.  **“黄金搭档”**：**熵权法 + TOPSIS** 是建模比赛中最经典的组合拳（先用熵权法客观求权，再用TOPSIS排名）。

---

### 四、 Python 代码框架

这份代码我做了一个**“增强版”**。它不仅仅是TOPSIS，如果用户不传权重，它会自动内部调用**熵权法**计算权重（即 **Entropy-TOPSIS** 模型），这是比赛最常用的形态。

```python
import pandas as pd
import numpy as np

def topsis(data, direction_list, weights=None):
    """
    TOPSIS 综合评价模型 (支持自动计算熵权)
    :param data: pandas DataFrame, 原始数据
    :param direction_list: list, 指标方向 (1:正向/越大越好, 0:负向/越小越好)
    :param weights: list or np.array, 权重列表。如果为 None，则自动使用熵权法计算。
    :return: pandas DataFrame (包含得分和排名)
    """
    df = data.copy().astype(float)
    n, m = df.shape

    # --- 1. 指标正向化处理 ---
    # 统一转化为“越大越好”
    for i, col in enumerate(df.columns):
        direction = direction_list[i]
        if direction == 0: # 负向指标 -> 正向转化
            # 使用 max - x 的方式转化
            df[col] = df[col].max() - df[col]
        # 如果是中间型或区间型，建议在传入函数前在 Excel 或外部处理好，保持代码简洁

    # --- 2. 数据标准化 (向量归一化) ---
    # TOPSIS 标准做法: x / sqrt(sum(x^2))
    norm_df = df / np.sqrt((df ** 2).sum())

    # --- 3. 确定权重 ---
    if weights is None:
        print("提示: 未传入权重，将自动使用【熵权法】计算客观权重...")
        # 临时使用 Min-Max 归一化来算熵权 (熵权法对负数敏感，向量归一化可能有问题，这里局部处理)
        # 注意：为了代码简便，这里直接用向量归一化的数据算熵权虽不严谨但可用，
        # 更严谨的做法是重新做一次 Min-Max 归一化算熵权。这里简化演示：

        # 简单处理：将 norm_df 归一化到 [0, 1] 用于算熵 (避免 log 负数或 0)
        temp_norm = (df - df.min()) / (df.max() - df.min())
        # 计算比重
        P = temp_norm / temp_norm.sum()
        P = P.fillna(0) # 处理分母为0
        # 计算熵
        k = 1 / np.log(n)
        E = -k * (P * np.log(P + 1e-6)).sum()
        # 差异系数
        D = 1 - E
        calc_weights = D / D.sum()
        weights = calc_weights.values
        print("自动计算的权重:", np.round(weights, 4))
    else:
        weights = np.array(weights)
        if len(weights) != m:
            raise ValueError("权重长度与指标列数不一致")

    # --- 4. 确定正负理想解 ---
    # 因为已经正向化了，所以正理想解就是 max，负理想解就是 min
    # 每一列的最大值
    Z_plus = norm_df.max()
    # 每一列的最小值
    Z_minus = norm_df.min()

    # --- 5. 计算距离 ---
    # 权重加权距离
    # 技巧: (x - max)^2 * w  <-- 这种写法是错的，应该是 sum( w * (x-max)^2 )
    # 或者先加权: weighted_data = data * w

    weighted_norm_df = norm_df * weights
    weighted_Z_plus = Z_plus * weights
    weighted_Z_minus = Z_minus * weights

    # 计算 D+ (与最优解的欧氏距离)
    # axis=1 表示按行求和
    D_plus = np.sqrt(((weighted_norm_df - weighted_Z_plus) ** 2).sum(axis=1))

    # 计算 D- (与最劣解的欧氏距离)
    D_minus = np.sqrt(((weighted_norm_df - weighted_Z_minus) ** 2).sum(axis=1))

    # --- 6. 计算得分 ---
    # S = D- / (D+ + D-)
    score = D_minus / (D_plus + D_minus)

    # 整理输出
    result = pd.DataFrame({
        'D+': D_plus,
        'D-': D_minus,
        '得分': score,
        '排名': score.rank(ascending=False)
    })

    return result

# ================= 使用示例 =================

if __name__ == "__main__":
    # 1. 准备数据
    # 案例：评价 4 款水质监测点的水质
    # 指标：含氧量(越高越好), PH值差异(越低越好), 细菌总数(越低越好), 矿物质(越高越好)
    data = {
        '含氧量': [7.2, 6.5, 8.1, 5.5],  # 正向 (1)
        'PH差值': [0.1, 0.5, 0.2, 0.8],  # 负向 (0) - 假设原始PH是7.0为最好，这里给出的是|PH-7|
        '细菌数': [120, 200, 50, 300],   # 负向 (0)
        '矿物质': [15, 12, 18, 10]       # 正向 (1)
    }
    df = pd.DataFrame(data, index=['A点', 'B点', 'C点', 'D点'])
    print("原始数据:\n", df)

    # 2. 定义方向
    directions = [1, 0, 0, 1]

    # 3. 运行模型 (不传 weights，让它自动用熵权法)
    print("-" * 30)
    print("模式1: 自动计算熵权 (Entropy-TOPSIS)")
    results_auto = topsis(df, directions, weights=None)
    print(results_auto.sort_values(by='排名'))

    # 4. 运行模型 (手动指定权重)
    # 比如 AHP 算出来权重是 [0.2, 0.3, 0.4, 0.1]
    print("-" * 30)
    print("模式2: 手动指定权重 (AHP-TOPSIS)")
    manual_weights = [0.2, 0.3, 0.4, 0.1]
    results_manual = topsis(df, directions, weights=manual_weights)
    print(results_manual.sort_values(by='排名'))
```

### 代码使用的“修修补补”指南：

1.  **关于“中间型”指标**：
    *   代码里只处理了**正向(1)**和**负向(0)**。
    *   如果你有一个指标是“PH值越接近7越好”，这是**中间型**。
    *   *怎么修？* **不要在代码里改**，太麻烦。建议在 Excel 里先处理好这一列。
        *   公式：$M = \max(|x_i - 7|)$
        *   转换后数据：$x'_{i} = 1 - \frac{|x_i - 7|}{M}$
        *   处理完这一列后，它就变成了**正向指标**（越大越好，最大是1），然后在代码 `directions` 里对应位置填 `1` 即可。

2.  **关于权重的选择**：
    *   **想省事/纯数据驱动**：直接传 `weights=None`，用熵权TOPSIS，这最客观。
    *   **有专家意见**：先用我之前给的 AHP 代码算出 `weights` 列表，然后传进去，这叫 AHP-TOPSIS。
    *   **既要又要**：可以将 AHP 算出的权重和 熵权法算出的权重，各乘 0.5 相加，作为综合权重传入。

3.  **结果为 NaN？**
    *   如果某一列数据**全部相同**（比如大家的体温都是36.5），分母的方差可能为0，导致报错。
    *   *解决办法*：把这一列删掉。既然大家这一项都一样，它就不具备评价意义了。